In [ ]:
!pip install natasha corus
import nltk
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import random
import numpy as np
import pandas as pd
import re

from corus import load_lenta
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('russian'))

import pymorphy2
morph = pymorphy2.MorphAnalyzer()

segmenter = Segmenter()
morph_tagger = NewsMorphTagger(NewsEmbedding())
morph_vocab = MorphVocab()
stop_words = set(stopwords.words('russian'))

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
!pip install pymorphy2 corus

import inspect
if not hasattr(inspect, 'getargspec'):
    import collections
    def getargspec(func):
        sig = inspect.signature(func)
        args = [
            p.name for p in sig.parameters.values()
            if p.kind == p.POSITIONAL_OR_KEYWORD
        ]
        varargs = None
        varkw = None
        defaults = tuple(
            p.default for p in sig.parameters.values()
            if p.default is not p.empty
        )
        return collections.namedtuple('ArgSpec', 'args varargs keywords defaults')(
            args, varargs, varkw, defaults
        )
    inspect.getargspec = getargspec


In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^а-яa-z ]+', ' ', text)
    words = text.split()
    lemmas = [morph.parse(w)[0].normal_form for w in words if w not in stop_words]
    return ' '.join(lemmas)


In [ ]:
url = 'https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz'
data = pd.read_csv(url, compression='gzip', usecols=['title', 'text', 'topic'])

sample_size = 100_000
data = data.sample(n=sample_size, random_state=RANDOM_STATE)
data['processed_text'] = (data['title'].fillna('') + ' ' + data['text'].fillna('')).apply(preprocess_text)
data = data[data['processed_text'].str.strip().astype(bool)]

# Удаление редких классов
class_counts = data['topic'].value_counts()
data = data[data['topic'].isin(class_counts[class_counts >= 2].index)]

# Кодирование классов
data['topic'] = data['topic'].astype('category').cat.codes


In [ ]:
data['processed_text'] = (data['title'].fillna('') + ' ' + data['text'].fillna('')).apply(preprocess_text)

for i, row in data['processed_text'].head(10).items():
    print(f"→ {row}")


→ egyptair объявить подорожание билет египетский перевозчик egyptair сообщить возможный повышение стоимость билет свой международный рейс девальвация национальный валюта такой заявление сделать генеральный директор перевозчик шериф фатхи слово приводить рамблер путешествие фатхи заверить пока компания принять решение признать рост неизбежный пока тариф перевозка обсуждаться сайт авиакомпания билет подорожать руководство авиакомпания это объяснить изменение налогообложение предполагаться повышение цена коснуться международный перелёт стоимость внутренний рейс остаться прежний отмечаться центральный банк египет девальвировать национальный валюта понедельник март процент египетский фунт доллар ранее сообщаться министр иностранный дело египет заявить аэропорт каир шарм эль шейх хургад ввести новый мера безопасность который рекомендовать российский эксперт указ приостановка полёт россия египет президент россия владимир путин подписать ноябрь это произойти катастрофа самолёт airbus a компани

In [ ]:
print(f"Размер после фильтрации: {data.shape}")
print("Пример строки:", data['processed_text'].iloc[0] if not data.empty else "Нет данных")


Размер после фильтрации: (99975, 4)
Пример строки: egyptair объявить подорожание билет египетский перевозчик egyptair сообщить возможный повышение стоимость билет свой международный рейс девальвация национальный валюта такой заявление сделать генеральный директор перевозчик шериф фатхи слово приводить рамблер путешествие фатхи заверить пока компания принять решение признать рост неизбежный пока тариф перевозка обсуждаться сайт авиакомпания билет подорожать руководство авиакомпания это объяснить изменение налогообложение предполагаться повышение цена коснуться международный перелёт стоимость внутренний рейс остаться прежний отмечаться центральный банк египет девальвировать национальный валюта понедельник март процент египетский фунт доллар ранее сообщаться министр иностранный дело египет заявить аэропорт каир шарм эль шейх хургад ввести новый мера безопасность который рекомендовать российский эксперт указ приостановка полёт россия египет президент россия владимир путин подписать ноябрь 

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['processed_text'], data['topic'], test_size=0.2, stratify=data['topic'], random_state=RANDOM_STATE)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.25, stratify=train_labels, random_state=RANDOM_STATE)


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(train_texts, train_labels)
dummy_preds = dummy.predict(val_texts)
print(f'Dummy Accuracy: {accuracy_score(val_labels, dummy_preds):.4f}')


Dummy Accuracy: 0.2188


In [ ]:
count_pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, solver='saga'))
])
count_pipeline.fit(train_texts, train_labels)
count_preds = count_pipeline.predict(val_texts)
print(f'CountVectorizer Accuracy: {accuracy_score(val_labels, count_preds):.4f}')


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


CountVectorizer Accuracy: 0.8064


In [ ]:
tfidf_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, max_iter=2000, solver='saga'))
])
tfidf_pipeline.fit(train_texts, train_labels)
tfidf_preds = tfidf_pipeline.predict(val_texts)
print(f'TFIDF Accuracy: {accuracy_score(val_labels, tfidf_preds):.4f}')


TFIDF Accuracy: 0.8114


In [ ]:
param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'classifier__C': [0.1, 1, 10]
}

gs = GridSearchCV(tfidf_pipeline, param_grid, cv=3, scoring='accuracy')
gs.fit(train_texts, train_labels)
print(f'Лучшие параметры: {gs.best_params_}')

best_model = gs.best_estimator_
test_preds = best_model.predict(test_texts)
print(f'Final Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}')


Лучшие параметры: {'classifier__C': 10, 'vectorizer__ngram_range': (1, 2)}
Final Test Accuracy: 0.8185
